## 文献调研SurveyAgent案例

本案例演示如何单独使用 `SurveyAgent` 对给定研究主题做多源文献检索与筛选。

### 1. 环境设置与模块导入

In [ ]:
import json
import os
import sys

sys.path.append("../../")

from mindscience_agent.config import MindScienceConfig
from mindscience_agent.agents.agent_factory import AgentFactory
from mindscience_agent.model.model_factory import ModelFactory

d:\anaconda3\envs\JiuwenClaw\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


### 2. Semantic Scholar API 密钥（环境变量 `S2_API_KEY`）

当 `mindscience_agent.yaml` 中 `tools.paper_survey.sources` 包含 `semantic_scholar` 时，底层会读取环境变量 **`S2_API_KEY`**。

推荐做法：在操作系统或终端中预先 `set` / `export S2_API_KEY=...`。若未设置，下面单元会使用 Notebook 内提供的默认值。

In [ ]:
# 若已在环境中配置 S2_API_KEY，则不会覆盖；否则使用下列默认值
os.environ.setdefault(
    "S2_API_KEY",
    "***",  # 用户自己的真实的密钥
)
print("S2_API_KEY configured:", bool(os.environ.get("S2_API_KEY")))

S2_API_KEY configured: True


### 3. 加载配置文件

加载实验配置文件，可以根据实际情况修改，**配置文件中需包含用户自己的模型 API 密钥等信息**。

文献工具由 **`tools.paper_survey`** 控制（`sources`、`max_results`）。本目录配置默认包含 **PubMed、arXiv、Semantic Scholar** 三源。

In [ ]:
config_path = "./mindscience_agent.yaml"
config = MindScienceConfig.init_config_from_yaml(config_path)

[INFO] 2026-04-10-16:31:31.809.000 [mindscience_agent\config\mindscience_config.py:301] 
MINDSCIENCEAGENT CONFIGURATION
├─ agents:
│   ├─ survey:
│   │   ├─ agent_type: survey
│   │   ├─ model_config:
│   │   │   ├─ model_name: kimi-k2.5
│   │   │   ├─ api_key: ***
│   │   │   ├─ base_url: https://dashscope.aliyuncs.com/compatible-mode/v1
│   │   │   ├─ provider: openai
│   │   │   ├─ temperature: 0.2
│   │   │   ├─ max_tokens: 4096
│   │   │   ├─ timeout: 60
│   │   │   ├─ max_retries: 2
│   │   │   └─ max_connections: 8
│   │   ├─ max_retries: 2
│   │   ├─ use_tool_retriever: False
│   │   ├─ skill_path: 
│   │   └─ max_papers: 5
│   ├─ plan:
│   │   ├─ agent_type: plan
│   │   ├─ model_config:
│   │   │   ├─ model_name: kimi-k2.5
│   │   │   ├─ api_key: ***
│   │   │   ├─ base_url: https://dashscope.aliyuncs.com/compatible-mode/v1
│   │   │   ├─ provider: openai
│   │   │   ├─ temperature: 0.7
│   │   │   ├─ max_tokens: 4096
│   │   │   ├─ timeout: 60
│   │   │   ├─ max_retries: 2
│

### 4. 初始化 SurveyAgent

通过 `AgentFactory` 创建 `SurveyAgent`，并注入 `ModelFactory` 生成的模型实例。`tool_config` 中的 `paper_survey` 与测试中 `_create_tool_config()` 一致，由 YAML 的 `tools` 段提供。

In [9]:
model_factory = ModelFactory()
survey_agent = AgentFactory.create_agent(
    agent_type="survey",
    config=config.get_agent_config("survey"),
    tool_config=config.tools,
    model_factory=model_factory,
)

survey_cfg = config.get_agent_config("survey")
print(
    "SurveyAgent ready:",
    "max_papers=", getattr(survey_cfg, "max_papers", None),
    "model=", survey_cfg.model_config.model_name,
)

[INFO] 2026-04-10-16:31:35.778.000 [mindscience_agent\agents\agent_factory.py:79] Created agent instance: survey (SurveyAgent)


SurveyAgent ready: max_papers= 5 model= kimi-k2.5


### 5. 定义调研问题

下面为**通用生物医学信息学 / 医疗 AI** 主题示例。

In [10]:
query = (
    "Survey recent machine learning methods for healthcare risk prediction and clinical decision support, "
    "with emphasis on interpretability and validation benchmarks; prioritize work from 2023–2025."
)
messages = [{"role": "user", "content": query}]

print("User query (first message content):\n", query)

User query (first message content):
 Survey recent machine learning methods for healthcare risk prediction and clinical decision support, with emphasis on interpretability and validation benchmarks; prioritize work from 2023–2025.


### 6. 执行文献调研并展示结果

调用 `await survey_agent.execute(messages)`。该过程包含多轮模型调用与外部检索，**可能耗时数分钟**。返回值为论文字典列表；下面打印篇数与前若干条的标题、来源与分数。

In [11]:
papers = await survey_agent.execute(messages)

print(f"Returned {len(papers)} papers\n")
for i, p in enumerate(papers[:10]):
    title = p.get("title", "")
    src = p.get("source", "")
    score = p.get("score", "")
    suf = "..." if len(title) > 120 else ""
    print(f"{i + 1}. [{src}] score={score} {title[:120]}{suf}")

Returned 5 papers

1. [arXiv] score=9 What is Interpretable? Using Machine Learning to Design Interpretable Decision-Support Systems
2. [arXiv] score=8 Enhancing clinical decision support with physiological waveforms -- a multimodal benchmark in emergency care
3. [arXiv] score=7 Personalized and Reliable Decision Sets: Enhancing Interpretability in Clinical Decision Support Systems
4. [semantic_scholar] score=8 Construction and validation of a risk prediction model for complications in patients with acute leukemia based on machin...
5. [semantic_scholar] score=7 Interpretable Machine Learning Framework for Diabetes Prediction: Integrating SMOTE Balancing with SHAP Explainability f...


### 7. 查看单条结果结构

In [12]:
if papers:
    print(json.dumps(papers[0], ensure_ascii=False, indent=2))
else:
    print("No papers returned.")

{
  "id": "6",
  "title": "What is Interpretable? Using Machine Learning to Design Interpretable Decision-Support Systems",
  "authors": [
    "Owen Lahav",
    "Nicholas Mastronarde",
    "Mihaela van der Schaar"
  ],
  "abstract": "Recent efforts in Machine Learning (ML) interpretability have focused on creating methods for explaining black-box ML models. However, these methods rely on the assumption that simple approximations, such as linear models or decision-trees, are inherently human-interpretable, which has not been empirically tested. Additionally, past efforts have focused exclusively on comprehension, neglecting to explore the trust component necessary to convince non-technical experts, such as clinicians, to utilize ML models in practice. In this paper, we posit that reinforcement learning (RL) can be used to learn what is interpretable to different users and, consequently, build their trust in ML models. To validate this idea, we first train a neural network to provide ris